# validating the dublin data using great expectations

## Imports

In [ ]:
import great_expectations as gx
from great_expectations.expectations.expectation import ExpectationConfiguration
import pandas as pd

## read the dublin parsed data in a data frame

In [ ]:
df_final = pd.read_csv('../data/processed/ppr_dublin_parsed.csv')

## focus on clean columns

In [ ]:
clean_cols = [
    'date_of_sale_ddmmyyyy',
    'county',
    'price_clean',
    'is_vat_exclusive',
    'StreetName',
    'Municipality'
]
# Create a focused dataframe for the assistant
df_clean_subset = df_final[clean_cols]

## use data assistant to profile the data

In [ ]:
# 1. init the context
context = gx.get_context()
# 2. connt the data frame as data asset
datasource = context.sources.add_pandas(name="property_source")
data_asset = datasource.add_dataframe_asset(name="dublin_area")
# 3. create the batch request
batch_request = data_asset.build_batch_request(dataframe=df_clean_subset)

In [ ]:
assistant_result = context.assistants.onboarding.run(
    batch_request=batch_request)

In [ ]:
assistant_result

In [ ]:
# extract the rules from assistant_result
suite = assistant_result.get_expectation_suite(
    expectation_suite_name="dublin_property_gold_contract")

In [ ]:
for expectation in suite.expectations:
    print(expectation.expectation_type)
    print(expectation.kwargs)
    print("-" * 20)

In [ ]:
price_expectations = [price_expectation for price_expectation in suite.expectations if price_expectation.kwargs.get(
    "column") == "price_clean"]
for price_expectation in price_expectations:
    print(f"rule type {price_expectation.expectation_type}")
    print(f"rule values {price_expectation.kwargs}")
    print("-"*15)

💡 looks a realistic price range is between 5000 to 300,000,000 we add this to rules

In [ ]:
price_expectation_config = ExpectationConfiguration(expectation_type="expect_column_values_to_be_between", kwargs={
                                                    'column': 'price_clean', 'max_value': 300000000, 'strict_min': False, 'min_value': 5000, 'mostly': 1.0, 'strict_max': False})
price_null_config = ExpectationConfiguration(
    expectation_type="expect_column_values_to_not_be_null", kwargs={'column': 'price_clean'})

In [ ]:
suite.add_expectation_configurations(
    [price_expectation_config, price_null_config], overwrite_existing=True)

In [ ]:
price_expectations = [price_expectation for price_expectation in suite.expectations if price_expectation.kwargs.get(
    "column") == "price_clean"]
for price_expectation in price_expectations:
    print(f"rule type {price_expectation.expectation_type}")
    print(f"rule values {price_expectation.kwargs}")
    print("-"*15)

In [ ]:
# Save the suite to a JSON file
context = context.convert_to_file_context()
context.save_expectation_suite(suite)

## Load the new json and run the validtor

In [ ]:
context = gx.get_context()
df_parsed = pd.read_csv("../data/processed/ppr_dublin_parsed.csv")
datasource = context.get_datasource("property_source")
data_asset = datasource.get_asset("dublin_area")
batch_request = data_asset.build_batch_request(dataframe=df_parsed)
validator = context.get_validator(
    batch_request=batch_request, expectation_suite_name="dublin_property_gold_contract")
results = validator.validate()

#View the summary
print(f"Validation Success: {results.success}")
print(f"Stats: {results.statistics}")